#Task 1: Database Design and Table Creation
We will first establish a connection to bookstore.db and create the three tables with the specified constraints.

In [1]:
import sqlite3
import pandas as pd

# 1. Create the database connection
conn = sqlite3.connect('bookstore.db')
cursor = conn.cursor()

# Enable Foreign Key support in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. Define and Create Tables
# Books Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER DEFAULT 0
)
''')

# Customers Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT
)
''')

# Orders Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    book_id INTEGER,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
    FOREIGN KEY (book_id) REFERENCES Books(book_id)
)
''')

# 4. Display Schema using PRAGMA
print("=== TASK 1: Database Creation ===")
tables = ['Books', 'Customers', 'Orders']
for table in tables:
    print(f"\nSchema for {table} table:")
    schema = cursor.execute(f"PRAGMA table_info({table})").fetchall()
    # Formatting for display similar to sample output
    df_schema = pd.DataFrame(schema, columns=['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk'])
    print(df_schema)

conn.commit()

=== TASK 1: Database Creation ===

Schema for Books table:
   cid            name     type  notnull dflt_value  pk
0    0         book_id  INTEGER        0       None   1
1    1           title     TEXT        1       None   0
2    2          author     TEXT        1       None   0
3    3           price     REAL        1       None   0
4    4  stock_quantity  INTEGER        0          0   0

Schema for Customers table:
   cid         name     type  notnull dflt_value  pk
0    0  customer_id  INTEGER        0       None   1
1    1         name     TEXT        1       None   0
2    2        email     TEXT        1       None   0
3    3         city     TEXT        0       None   0
4    4    join_date     TEXT        0       None   0

Schema for Orders table:
   cid          name     type  notnull dflt_value  pk
0    0      order_id  INTEGER        0       None   1
1    1   customer_id  INTEGER        0       None   0
2    2       book_id  INTEGER        0       None   0
3    3      quan

#Task 2: Data Insertion and Querying
We use executemany() for efficiency. Note that for Orders, the input data needs to match the column order we defined.

In [2]:
# Part A: Insert Data
books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)
]

customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]

orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]

cursor.executemany("INSERT INTO Books (title, author, price, stock_quantity) VALUES (?, ?, ?, ?)", books_data)
cursor.executemany("INSERT INTO Customers (name, email, city, join_date) VALUES (?, ?, ?, ?)", customers_data)
cursor.executemany("INSERT INTO Orders (customer_id, book_id, quantity, order_date, total_amount) VALUES (?, ?, ?, ?, ?)", orders_data)

conn.commit()

# Part B: Complex Queries
print("\n=== TASK 2: Data Insertion and Querying ===")

# 1. Customers from Mumbai
print("\nCustomers from Mumbai:")
print(pd.read_sql("SELECT * FROM Customers WHERE city = 'Mumbai'", conn))

# 2. Books price > 800 and stock > 10
print("\nBooks priced > 800 with stock > 10:")
print(pd.read_sql("SELECT * FROM Books WHERE price > 800 AND stock_quantity > 10", conn))

# 4. Customer with most orders
print("\nCustomer with the most orders:")
query_most_orders = '''
SELECT c.name, COUNT(o.order_id) as order_count
FROM Customers c
JOIN Orders o ON c.customer_id = o.customer_id
GROUP BY c.name
ORDER BY order_count DESC
LIMIT 1
'''
print(pd.read_sql(query_most_orders, conn))

# 5. Total Revenue
total_revenue = cursor.execute("SELECT SUM(total_amount) FROM Orders").fetchone()[0]
print(f"\nTotal Revenue: {total_revenue}")


=== TASK 2: Data Insertion and Querying ===

Customers from Mumbai:
   customer_id          name             email    city   join_date
0            1  Rahul Sharma   rahul@email.com  Mumbai  2024-01-15
1            5  Vikram Singh  vikram@email.com  Mumbai  2024-02-15

Books priced > 800 with stock > 10:
   book_id                  title    author  price  stock_quantity
0        2  Data Science Handbook  Jane Doe  899.5              15

Customer with the most orders:
           name  order_count
0  Rahul Sharma            2

Total Revenue: 7994.96


#Task 3: Pandas Integration and Analysis
This part demonstrates moving data from SQL to Pandas for reporting and pushing Pandas DataFrames back into SQL.

In [3]:
print("\n=== TASK 3: Pandas Integration and Analysis ===")

# Part A: SQL to Pandas
df_books = pd.read_sql("SELECT * FROM Books", conn)
df_customers = pd.read_sql("SELECT * FROM Customers", conn)
df_orders = pd.read_sql("SELECT * FROM Orders", conn)

# Pandas Merge for Order Report
report = df_orders.merge(df_customers, on='customer_id').merge(df_books, on='book_id')
final_report = report[['order_id', 'name', 'city', 'title', 'quantity', 'total_amount']]
final_report.columns = ['Order ID', 'Customer Name', 'Customer City', 'Book Title', 'Quantity', 'Total Amount']

print("\nComprehensive Order Report (First 3 rows):")
print(final_report.head(3))

# Part B: Pandas to SQL
discounts_data = {
    'book_id': [1, 2, 3, 4, 5],
    'discount_percent': [10, 15, 5, 20, 12]
}
df_discounts = pd.DataFrame(discounts_data)

# Save to new table
df_discounts.to_sql('discounts', conn, if_exists='replace', index=False)

# Join query for discounted prices
query_discount = '''
SELECT b.title, b.price, d.discount_percent,
       (b.price * (1 - d.discount_percent/100.0)) AS discounted_price
FROM Books b
JOIN discounts d ON b.book_id = d.book_id
'''
print("\nBook Titles with Discounted Prices:")
print(pd.read_sql(query_discount, conn))

conn.close()


=== TASK 3: Pandas Integration and Analysis ===

Comprehensive Order Report (First 3 rows):
   Order ID Customer Name Customer City             Book Title  Quantity  \
0         1  Rahul Sharma        Mumbai     Python Programming         2   
1         2  Rahul Sharma        Mumbai  Data Science Handbook         1   
2         3   Priya Patel         Delhi     Python Programming         1   

   Total Amount  
0       1199.00  
1        899.50  
2        599.99  

Book Titles with Discounted Prices:
                     title    price  discount_percent  discounted_price
0       Python Programming   599.99                10           539.991
1    Data Science Handbook   899.50                15           764.575
2  Machine Learning Basics  1299.00                 5          1234.050
3           SQL Essentials   499.99                20           399.992
4          Web Development   799.00                12           703.120
